# Support Vector Machine (SVM) - Breast Cancer Wisconsin Analizi

**Veri Seti:** `data.csv` (569 örnek, 30 sayısal öznitelik)

**Amaç:** Hücre çekirdeği ölçümlerine bakarak tümörün **diagnosis** (iyi huylu / kötü huylu) sınıflandırmasını yapmak.

## SVM Teorisi

**Support Vector Machine**, sınıflar arasındaki **marjini (boşluğu)** maksimize eden bir ayırıcı hiper-düzlem bulmaya çalışan bir sınıflandırma algoritmasıdır.

### Temel Fikir
- Veriyi iki (veya daha fazla) sınıfa ayıran sonsuz sayıda doğru/düzlem çizilebilir.
- SVM, sınıflara en yakın noktalara (**destek vektörleri**) olan mesafeyi maksimize eden düzlemi seçer.
- Bu, modelin yeni verilerde daha iyi genelleme yapmasını sağlar.

### Kernel Trick
Veriler doğrusal olarak ayrılamıyorsa, **kernel fonksiyonları** kullanılarak veri daha yüksek boyutlu bir uzaya taşınır ve orada doğrusal olarak ayrılabilir hale getirilir:
- **Linear kernel:** K(x, x') = x·x' — doğrusal ayrım
- **RBF (Radial Basis Function) kernel:** K(x, x') = exp(-γ‖x-x'‖²) — doğrusal olmayan, yerel ilişkileri yakalar
- **Polynomial kernel:** K(x, x') = (x·x' + c)^d — polinom dereceli ilişkiler

### C ve Gamma Parametreleri
- **C:** Hata toleransını kontrol eder. Küçük C daha geniş marj + daha fazla hataya izin verir (daha az overfitting); büyük C daha dar marj + eğitim verisine daha sıkı uyum sağlar.
- **gamma:** RBF/poly kernelde tek bir eğitim örneğinin etki alanını belirler. Küçük gamma geniş etki alanı, büyük gamma dar/yerel etki alanı anlamına gelir.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 1. Veri Setini Kesfetme ve Degiskenleri Tanima

Asagida veri setindeki tüm degiskenleri (sutunlari) ve özelliklerini inceleyecegiz.

In [2]:
df = pd.read_csv('data.csv')

df = df.drop(columns=[c for c in df.columns if 'Unnamed' in c])
df = df.drop(columns=['id'])
df = df.rename(columns={'diagnosis': 'diagnosis'})
df['diagnosis'] = df['diagnosis'].map({'M': 'malignant', 'B': 'benign'})

print('=' * 70)
print('VERI SETI BOYUTU')
print('=' * 70)
print(f'Satir sayisi: {df.shape[0]}')
print(f'Sutun sayisi: {df.shape[1]}')
print()

print('=' * 70)
print('SUTUNLAR VE VERI TIPLERI')
print('=' * 70)
for col in df.columns:
    print(f"{col:<28} {str(df[col].dtype):<10}")

VERI SETI BOYUTU
Satir sayisi: 569
Sutun sayisi: 31

SUTUNLAR VE VERI TIPLERI
diagnosis                    str       
radius_mean                  float64   
texture_mean                 float64   
perimeter_mean               float64   
area_mean                    float64   
smoothness_mean              float64   
compactness_mean             float64   
concavity_mean               float64   
concave points_mean          float64   
symmetry_mean                float64   
fractal_dimension_mean       float64   
radius_se                    float64   
texture_se                   float64   
perimeter_se                 float64   
area_se                      float64   
smoothness_se                float64   
compactness_se               float64   
concavity_se                 float64   
concave points_se            float64   
symmetry_se                  float64   
fractal_dimension_se         float64   
radius_worst                 float64   
texture_worst                float64   
pe

In [3]:
print('=' * 70)
print('DEGISKEN GRUPLARI VE ACIKLAMALARI')
print('=' * 70)
variable_groups = {
    'mean *': 'Hücre çekirdeklerinin ortalama ölçümleri (radius, texture, perimeter, area, ...)',
    'error *': 'Ölçümlerin standart hatası (variasyon)',
    'worst *': 'Ölçümlerin en kötü (en büyük) değerleri',
    'diagnosis': 'Hedef değişken: malignant (kötü huylu) / benign (iyi huylu)'
}
for k, v in variable_groups.items():
    print(f'{k:<12}: {v}')

DEGISKEN GRUPLARI VE ACIKLAMALARI
mean *      : Hücre çekirdeklerinin ortalama ölçümleri (radius, texture, perimeter, area, ...)
error *     : Ölçümlerin standart hatası (variasyon)
worst *     : Ölçümlerin en kötü (en büyük) değerleri
diagnosis   : Hedef değişken: malignant (kötü huylu) / benign (iyi huylu)


In [4]:
print('=' * 70)
print('HEDEF DEGISKEN DAGILIMI')
print('=' * 70)
print(df['diagnosis'].value_counts())
print()
print((df['diagnosis'].value_counts(normalize=True) * 100).round(2))

HEDEF DEGISKEN DAGILIMI
diagnosis
benign       357
malignant    212
Name: count, dtype: int64

diagnosis
benign       62.74
malignant    37.26
Name: proportion, dtype: float64


In [5]:
print('=' * 70)
print('SAYISAL DEGISKENLERIN ISTATISTIKLERI (ilk 6 sutun)')
print('=' * 70)
print(df.iloc[:, :6].describe().to_string())

SAYISAL DEGISKENLERIN ISTATISTIKLERI (ilk 6 sutun)
       radius_mean  texture_mean  perimeter_mean    area_mean  smoothness_mean
count   569.000000    569.000000      569.000000   569.000000       569.000000
mean     14.127292     19.289649       91.969033   654.889104         0.096360
std       3.524049      4.301036       24.298981   351.914129         0.014064
min       6.981000      9.710000       43.790000   143.500000         0.052630
25%      11.700000     16.170000       75.170000   420.300000         0.086370
50%      13.370000     18.840000       86.240000   551.100000         0.095870
75%      15.780000     21.800000      104.100000   782.700000         0.105300
max      28.110000     39.280000      188.500000  2501.000000         0.163400


## 2. Veri Ön Isleme (Preprocessing)

SVM mesafe tabanli bir algoritma oldugu için:
1. **Hedef degisken** sayisala çevrilmeli (LabelEncoder)
2. **Sayisal degiskenler** ölçeklendirilmeli (StandardScaler)
3. **Egitim/test** ayrimi yapilmali

In [6]:
feature_cols = [c for c in df.columns if c != 'diagnosis']
X = df[feature_cols].copy()
y = df['diagnosis']

print('KULLANILACAK BAGIMSIZ DEGISKEN SAYISI:', len(feature_cols))
for i, col in enumerate(feature_cols, 1):
    print(f'{i:>2}. {col}')

KULLANILACAK BAGIMSIZ DEGISKEN SAYISI: 30
 1. radius_mean
 2. texture_mean
 3. perimeter_mean
 4. area_mean
 5. smoothness_mean
 6. compactness_mean
 7. concavity_mean
 8. concave points_mean
 9. symmetry_mean
10. fractal_dimension_mean
11. radius_se
12. texture_se
13. perimeter_se
14. area_se
15. smoothness_se
16. compactness_se
17. concavity_se
18. concave points_se
19. symmetry_se
20. fractal_dimension_se
21. radius_worst
22. texture_worst
23. perimeter_worst
24. area_worst
25. smoothness_worst
26. compactness_worst
27. concavity_worst
28. concave points_worst
29. symmetry_worst
30. fractal_dimension_worst


In [7]:
le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)

print('HEDEF DEGISKEN SAYISALLASTIRILDI')
for i, cls in enumerate(le_y.classes_):
    print(f'  {cls} -> {i}')

HEDEF DEGISKEN SAYISALLASTIRILDI
  benign -> 0
  malignant -> 1


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print('VERI AYRIMI (Train/Test)')
print(f'  Egitim seti  : {X_train.shape[0]} ornek (%80)')
print(f'  Test seti    : {X_test.shape[0]} ornek (%20)')
print(f'  Stratify     : Evet (sinif oranlari korunuyor)')

VERI AYRIMI (Train/Test)
  Egitim seti  : 455 ornek (%80)
  Test seti    : 114 ornek (%20)
  Stratify     : Evet (sinif oranlari korunuyor)


In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('VERI OLCEKLENDIRILDI (StandardScaler)')
print(f'  Ortalama (her sutun icin): 0')
print(f'  Standart sapma (her sutun icin): 1')
print(f'  X_train_scaled boyutu: {X_train_scaled.shape}')

VERI OLCEKLENDIRILDI (StandardScaler)
  Ortalama (her sutun icin): 0
  Standart sapma (her sutun icin): 1
  X_train_scaled boyutu: (455, 30)


## 3. SVM Modelleri

3 farkli kernel ile SVM egitip karsilastiracagiz:
- **Linear SVM** (dogrusal)
- **RBF SVM** (radyal tabanli - varsayilan)
- **Polynomial SVM** (polinom)

Ardindan **GridSearchCV** ile en iyi parametreleri bulup modeli optimize edecegiz.

In [10]:
print('=' * 50)
print('LINEAR SVM')
print('=' * 50)
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_linear):.4f}')
print(f'Destek vektor sayisi: {svm_linear.n_support_}')

LINEAR SVM
Test dogrulugu: 0.9649
Destek vektor sayisi: [19 19]


In [11]:
print('=' * 50)
print('RBF SVM (Varsayilan)')
print('=' * 50)
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_test_scaled)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_rbf):.4f}')
print(f'Destek vektor sayisi: {svm_rbf.n_support_}')

RBF SVM (Varsayilan)
Test dogrulugu: 0.9737
Destek vektor sayisi: [50 57]


In [12]:
print('=' * 50)
print('POLINOMIAL SVM (degree=3)')
print('=' * 50)
svm_poly = SVC(kernel='poly', degree=3, random_state=42)
svm_poly.fit(X_train_scaled, y_train)
y_pred_poly = svm_poly.predict(X_test_scaled)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_poly):.4f}')

POLINOMIAL SVM (degree=3)
Test dogrulugu: 0.8860


## 4. Hiperparametre Optimizasyonu (GridSearchCV)

En iyi `C` ve `gamma` degerlerini bulmak için **GridSearchCV** kullaniyoruz.

- **C**: [0.1, 1, 10, 100]
- **gamma**: ['scale', 'auto', 0.1, 0.01]
- **kernel**: ['rbf']
- **Cross-validation**: 5-fold

In [13]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'kernel': ['rbf']
}

print('ARANAN PARAMETRELER:')
for k, v in param_grid.items():
    print(f'  {k}: {v}')
print(f'Toplam kombinasyon: {len(param_grid["C"]) * len(param_grid["gamma"])}')

ARANAN PARAMETRELER:
  C: [0.1, 1, 10, 100]
  gamma: ['scale', 'auto', 0.1, 0.01]
  kernel: ['rbf']
Toplam kombinasyon: 16


In [14]:
grid_search = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

print(f'\n{"=" * 50}')
print('EN IYI PARAMETRELER')
print('=' * 50)
print(f'C          : {grid_search.best_params_["C"]}')
print(f'gamma      : {grid_search.best_params_["gamma"]}')
print(f'kernel     : {grid_search.best_params_["kernel"]}')
print(f'CV skoru   : {grid_search.best_score_:.4f}')

Fitting 5 folds for each of 16 candidates, totalling 80 fits



EN IYI PARAMETRELER
C          : 1
gamma      : scale
kernel     : rbf
CV skoru   : 0.9758


In [15]:
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)

print('=' * 50)
print('EN IYI MODEL SONUCLARI')
print('=' * 50)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Cross-validation skoru: {grid_search.best_score_:.4f}')
print()
print(classification_report(y_test, y_pred_best, target_names=le_y.classes_))

EN IYI MODEL SONUCLARI
Test dogrulugu: 0.9737
Cross-validation skoru: 0.9758

              precision    recall  f1-score   support

      benign       0.96      1.00      0.98        72
   malignant       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



In [16]:
print('=' * 50)
print('KARISIKLIK MATRISI (Confusion Matrix)')
print('=' * 50)
cm = confusion_matrix(y_test, y_pred_best)
print('Satirlar: GERCEK deger | Sutunlar: TAHMIN edilen deger')
print()
header = ' ' * 18 + ' '.join(f'{c:>14}' for c in le_y.classes_)
print(header)
print('-' * len(header))
for i, row in enumerate(cm):
    print(f'{le_y.classes_[i]:<18}' + ' '.join(f'{v:>14}' for v in row))

KARISIKLIK MATRISI (Confusion Matrix)
Satirlar: GERCEK deger | Sutunlar: TAHMIN edilen deger

                          benign      malignant
-----------------------------------------------
benign                        72              0
malignant                      3             39


In [17]:
print('=' * 50)
print('DESTEK VEKTOR ANALIZI')
print('=' * 50)
print(f'Toplam destek vektor sayisi  : {best_svm.n_support_.sum()}')
print(f'Toplam egitim ornegi         : {len(X_train_scaled)}')
print(f'Destek vektor orani          : {best_svm.n_support_.sum() / len(X_train_scaled) * 100:.1f}%')
print(f'Sinif basina destek vektor   : {dict(zip(le_y.classes_, best_svm.n_support_))}')

DESTEK VEKTOR ANALIZI
Toplam destek vektor sayisi  : 107
Toplam egitim ornegi         : 455
Destek vektor orani          : 23.5%
Sinif basina destek vektor   : {'benign': np.int32(50), 'malignant': np.int32(57)}


In [18]:
print('=' * 50)
print('ORNEK TAHMIN (Test Setinden 10 Kayit)')
print('=' * 50)
print(f'{"No":<5} {"Tahmin":<14} {"Gercek":<14} {"Dogru?":<8}')
print('-' * 45)
for i in range(10):
    tahmin = le_y.inverse_transform([best_svm.predict(X_test_scaled[i].reshape(1, -1))[0]])[0]
    gercek = le_y.inverse_transform([y_test[i]])[0]
    dogru = 'Evet' if tahmin == gercek else 'Hayir'
    print(f'{i:<5} {tahmin:<14} {gercek:<14} {dogru:<8}')

ORNEK TAHMIN (Test Setinden 10 Kayit)
No    Tahmin         Gercek         Dogru?  
---------------------------------------------
0     benign         benign         Evet    
1     malignant      malignant      Evet    
2     benign         benign         Evet    
3     benign         malignant      Hayir   
4     benign         benign         Evet    
5     benign         benign         Evet    
6     malignant      malignant      Evet    
7     benign         benign         Evet    
8     benign         benign         Evet    
9     benign         benign         Evet    


## 5. Sonuc ve Degerlendirme

| Model | Aciklama |
|-------|----------|
| Linear SVM | Dogrusal sinir; ozellikler zaten ayirt edici oldugundan yuksek basari verebilir |
| RBF SVM (varsayilan) | Dogrusal olmayan iliskileri yakalar, genelde en iyi/rekabetci sonucu verir |
| Polynomial SVM (3.derece) | Bazi veri setlerinde asiri uyum veya yetersiz uyuma yol acabilir |
| GridSearchCV ile optimize RBF | En iyi C ve gamma kombinasyonuyla genelde en yuksek dogrulugu saglar |

**Genel Yorum:** Breast Cancer Wisconsin veri seti dusuk gurultulu ve ozellikleri hedefle güçlü iliskili oldugundan SVM modelleri genelde yuksek dogruluk (%95+) elde eder. Destek vektor orani, modelin veriye ne kadar "genis marjla" uydugunun bir göstergesidir; düşük oran daha net bir ayrım, yüksek oran ise sınıflar arasında daha fazla örtüşme olduğunu gösterir.

In [19]:
print('=' * 50)
print('MODEL KARSILASTIRMA TABLOSU')
print('=' * 50)
comparison = pd.DataFrame({
    'Model': ['Linear SVM', 'RBF SVM', 'Polynomial SVM', 'GridSearchCV (En Iyi RBF)'],
    'Test Dogrulugu': [
        accuracy_score(y_test, y_pred_linear),
        accuracy_score(y_test, y_pred_rbf),
        accuracy_score(y_test, y_pred_poly),
        accuracy_score(y_test, y_pred_best)
    ]
})
comparison['Test Dogrulugu (%)'] = (comparison['Test Dogrulugu'] * 100).round(2)
display(comparison)

MODEL KARSILASTIRMA TABLOSU


,Model,Test Dogrulugu,Test Dogrulugu (%)
0,Linear SVM,0.964912,96.49
1,RBF SVM,0.973684,97.37
2,Polynomial SVM,0.885965,88.60
3,GridSearchCV (En Iyi RBF),0.973684,97.37
